In [1]:
from huggingface_hub import HfFileSystem
from huggingface_hub import snapshot_download
from multiprocess import Pool
import tarfile
import itertools
from tqdm import tqdm
from glob import glob
from collections import defaultdict
import random
import os

fs = HfFileSystem()

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!rm -rf Emilia-YODAS

In [3]:
files = fs.glob("datasets/amphion/Emilia-Dataset/Emilia-YODAS/*/*.tar")

In [4]:
languages = defaultdict(list)
for f in files:
    f = f.split('Emilia-Dataset/')[1]
    splitted = os.path.split(f)
    languages[splitted[0]].append(f)

In [5]:
for k, v in languages.items():
    print(k, len(v))

Emilia-YODAS/DE 161
Emilia-YODAS/EN 1362
Emilia-YODAS/FR 213
Emilia-YODAS/JA 30
Emilia-YODAS/KO 208
Emilia-YODAS/ZH 9


In [6]:
sample_languages = {k: random.sample(v, min(len(v), 200)) for k, v in languages.items()}
for k, v in sample_languages.items():
    print(k, len(v))

Emilia-YODAS/DE 161
Emilia-YODAS/EN 200
Emilia-YODAS/FR 200
Emilia-YODAS/JA 30
Emilia-YODAS/KO 200
Emilia-YODAS/ZH 9


In [8]:
for k, v in sample_languages.items():
    snapshot_download(
        repo_id="amphion/Emilia-Dataset", 
        repo_type="dataset",
        allow_patterns=v,
        local_dir="./",
        max_workers=20,
    )

Fetching 9 files: 100%|██████████| 9/9 [00:04<00:00,  1.84it/s]


In [10]:
files = glob('Emilia-YODAS/*/*.tar')
len(files)

800

In [11]:
def loop(files):
    files, _ = files
    for f in tqdm(files):
        try:
            with tarfile.open(f, "r") as tar:
                tar.extractall(path=os.path.split(f)[0])
            os.remove(f)
        except Exception as e:
            print(e)

In [12]:
!chmod -R 777 Emilia-YODAS

In [13]:
multiprocessing(files, loop, cores = min(len(files), 20), returned = False)

100%|██████████| 40/40 [30:44<00:00, 46.12s/it]
